# coarse

## Import

In [ ]:
import os
import gc
import sys
import warnings
import numpy as np
import pandas as pd
from glob import glob

sys.path.append("../input/pythonbox")
from box import Box

# Torch
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.io import read_image
from torch.utils.data import DataLoader, Dataset 
from torchvision.models import efficientnet_v2_l

# Lightning
import pytorch_lightning as pl
from pytorch_lightning import LightningDataModule, LightningModule, seed_everything

In [ ]:
!pip uninstall timm --yes

sys.path.append("/kaggle/input/timmmaster")
# import timm
from timm import create_model

# print(timm.__version__)

## Config

In [ ]:
config = {'exp_name':'exp_009',
          'root': '../input/petfinder-pawpularity-score/',  # Data root
          'seed': 2023,
          'n_splits': 5,
          'n_epochs': 50,
          'early_stop': 10,
          'image_size': 450,
          'lr': 1e-4,  # 如果有使用lr_find這個值會自動更改
          'lr_find': {'max_lr': 1e-3,
                      'min_lr': 1e-6,
                      'num_training': 100},
          'model':{
              'package': 'torchvision',  # timm or torchvision
              'name': 'efficientnet_v2_l',
              'output_dim': 1,
              'pretrain': False,
          },
          'save_dir': 'efficientnet_v2_l',  # 儲存權重與log的資料夾
          'train_loader': {
              'batch_size': 16,
              'shuffle': True,
              'num_workers': os.cpu_count(),
              'pin_memory': True,
              'drop_last': False
          },
          'val_loader': {
              'batch_size': 16,
              'shuffle': False,
              'num_workers': os.cpu_count(),
              'pin_memory': True,
              'drop_last': False
          },
          'test_loader': {
              'batch_size': 8,
              'shuffle': False,
              'num_workers': os.cpu_count(),
              'pin_memory': False,
              'drop_last': False
          },
          'loss': 'RMSE',
}

config = Box(config)

## Fix Seed

In [ ]:
seed_everything(config.seed)

## Tools

In [ ]:
def mixup(x: torch.Tensor, y: torch.Tensor, alpha: float = 1.0):
    assert alpha > 0, "alpha should be larger than 0"
    assert x.size(0) > 1, "Mixup cannot be applied to a single instance."

    lam = np.random.beta(alpha, alpha)
    rand_index = torch.randperm(x.size()[0])
    mixed_x = lam * x + (1 - lam) * x[rand_index, :]
    target_a, target_b = y, y[rand_index]
    return mixed_x, target_a, target_b, lam


def RMSE(predict,target):
    return torch.sqrt(nn.MSELoss()(predict.float(), target.float()))

def MSE(predict,target):
    return nn.MSELoss()(predict.float(), target.float())

## Dataset

In [ ]:
class PetfinderDataset(Dataset):
    """Dataset
    Args:
        df: the dataframe from csv, and the "Id" column needs to be the path of Image
    """
    def __init__(self, df, transform=None, image_size=224):
        
        self._X = df["Id"].values
        self._y = None
        self.transform = transform
        
        # 判斷有沒有分數
        if "Pawpularity" in df.keys():
            self._y = df["Pawpularity"].values
            
    def __len__(self):
        return len(self._X)

    def __getitem__(self, idx):
        image_path = self._X[idx]
        image = read_image(image_path)
        image = self.transform(image)
        
        if self._y is not None:
            label = self._y[idx]
            return image, label
        return image

## Model

In [ ]:
class Model(pl.LightningModule):
    def __init__(self, hparams):
        super().__init__()
        self.save_hyperparameters(hparams)  # 儲存超參數
        
        if 'nn' in self.hparams.loss:
            self._criterion = eval(self.hparams.loss)()
        else:
            self._criterion = eval(self.hparams.loss)
        self.metrics = RMSE
        
        self.validation_step_outputs = {'val/logits': [],
                                        'val/pred': [],
                                        'val/labels': []}  # 用來計算epoch的val/loss, val/rmse
        
        self.__build_model()
        
    def __build_model(self):
        if self.hparams.model.package == 'timm':
            self.backbone = create_model(self.hparams.model.name,
                                         pretrained=False, 
                                         num_classes=0, 
                                         in_chans=3)
            num_features = self.backbone.num_features
        
        elif self.hparams.model.package == 'torchvision':
            # weights = 'DEFAULT' if self.hparams.model.pretrain else None
            self.backbone = eval(self.hparams.model.name)(weights=None)
            num_features = 1000
            
        self.fc = nn.Sequential(nn.Dropout(0.5), 
                                nn.Linear(num_features, 
                                          self.hparams.model.output_dim))

    def forward(self, x):
        f = self.backbone(x)
        out = self.fc(f)
        return out

    def training_step(self, batch, batch_idx):
        images, labels = batch
        
        # Mixup (50%的機率)
        if torch.rand(1)[0] < 0.5:
            mix_images, target_a, target_b, lam = mixup(images, labels, alpha=0.5)
            logits = self(mix_images).squeeze().sigmoid()
            loss = self._criterion(logits, target_a) * lam + (1 - lam) * self._criterion(logits, target_b)
        else:
            logits = self(images).squeeze().sigmoid()
            loss = self._criterion(logits, labels)
            
        self.log("train/loss", loss, prog_bar=True)
        return loss
        
    def validation_step(self, batch, batch_idx):
        images, labels = batch
        
        logits = self(images).squeeze().sigmoid()
        loss = self._criterion(logits, labels)
        
        self.validation_step_outputs['val/logits'].append(logits)
        self.validation_step_outputs['val/labels'].append(labels)
        
        return {'val/loss': loss}
    
    def on_validation_epoch_end(self):
        logits = torch.cat(self.validation_step_outputs['val/logits'], dim=0)
        labels = torch.cat(self.validation_step_outputs['val/labels'], dim=0)
        pred = logits
        loss = self._criterion(logits, labels)
        metric = self.metrics(pred, labels) * 100.
        
        self.log('val/loss', loss, prog_bar=True)
        self.log('val/metric', metric, prog_bar=True)
        
        self.validation_step_outputs['val/logits'].clear()
        self.validation_step_outputs['val/pred'].clear()
        self.validation_step_outputs['val/labels'].clear()

    def predict_step(self, batch, batch_idx):
        images = batch
        pred = self(images).sigmoid()  # return後會自動轉成numpy
        if pred.dim == 3:
            pred = pred.squeeze()
        return pred
    
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        return optimizer

## Test

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]  # RGB
IMAGENET_STD = [0.229, 0.224, 0.225]  # RGB

test_transform = T.Compose([T.Resize(config.image_size),
                            T.CenterCrop([config.image_size, config.image_size]),
                            T.ConvertImageDtype(torch.float),
                            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])

stage = 'test'
df = pd.read_csv(os.path.join(config.root, stage+'.csv'))
df["Id"] = df["Id"].apply(lambda x: os.path.join(config.root, stage, x + ".jpg")) # 將ID改成圖片路徑
if "Pawpularity" in df.keys():
    test_y = df["Pawpularity"].astype(float).apply(lambda x: x / 100.).to_numpy() # 將Pawpularity軟換到[0, 1]
    df.drop('Pawpularity', axis=1, inplace=True)

# df to dataset
predict_data = PetfinderDataset(df, test_transform, config.image_size)
predict_loader = DataLoader(predict_data, **config.test_loader)

### kFold predictions

In [ ]:
warnings.filterwarnings("ignore")  # 關閉 Warning
torch.set_float32_matmul_precision('high')  # 設置高精度(根據顯卡調整)

total_predictions = []
for fold in range(config.n_splits):
    model_weight = glob(f'/kaggle/input/{config.save_dir}/fold_{fold}/version_0/*.ckpt')[0]

    model = Model(config).load_from_checkpoint(model_weight, , map_location='cuda:0')

    trainer = pl.Trainer(logger=False)
    predictions = trainer.predict(model, dataloaders=predict_loader)
    
    total_predictions.append(np.concatenate(predictions).flatten())
    
    # 清除變數與快取
    del model, trainer, predictions
    torch.cuda.empty_cache()
    gc.collect()

### 計算平均預測分數

In [ ]:
mean_predictions = np.array(total_predictions).mean(axis=0)
print(mean_predictions)

### 計算RMSE

In [ ]:
# RMSE(torch.tensor(mean_predictions), torch.tensor(test_y)).item()

### 儲存平均預測分數

In [ ]:
# 儲存csv
df = pd.read_csv(os.path.join(config.root, stage+'.csv'))
df_id = df['Id']

mean_predicts_df = pd.DataFrame(mean_predictions * 100., columns=["Pawpularity"])
df = pd.concat([df_id, mean_predicts_df], axis=1)
df.to_csv("submission.csv", index=False)
df.head(5)